# TMC-gray-recon 实拍数据推理

这个 notebook 展示了如何使用训练好的 `SDGrayNet` 模型对实拍的 `.tmdat` 数据进行灰度重建。可视化参考了 denoise 项目的风格。

In [ ]:
import os, sys, torch, cv2, math
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F

from model import SDGrayNet
from dataset import upsample_sd
from tianmoucv.data import TianmoucDataReader
from tianmoucv.isp import vizDiff

%load_ext autoreload
%autoreload 2

## 1. 加载模型

In [ ]:
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 初始化模型 (2输入通道, 1输出通道)
model = SDGrayNet(inChannels=2, outChannels=1).to(device)

# 加载指定的权重
ckpt_path = "/home/lyh/TMC-gray-recon/ckpts_tmp/2026-06-05_22-29-33/best_model.pth"
if os.path.exists(ckpt_path):
    state_dict = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state_dict)
    print(f"Model loaded from: {ckpt_path}")
else:
    raise FileNotFoundError(f"Checkpoint not found at {ckpt_path}")

model.eval()

## 2. 设置实拍数据路径

In [ ]:
# 实拍数据路径 (请根据实际情况修改)
datapath = "/data1/lyh/tianmoucData_fast/tianmoucReconDataset_0701new/test/extreme"
if not os.path.exists(datapath):
    print(f"[WARNING] Path {datapath} does not exist. Please update the path.")

results_dir = "./inference_results_real"
os.makedirs(results_dir, exist_ok=True)

def normalize_for_viz(x):
    """逐样本归一化到 [0, 1] 用于显示"""
    x_min = x.min()
    x_max = x.max()
    return (x - x_min) / (x_max - x_min + 1e-8)

## 3. 执行推理与对比可视化

In [ ]:
num_samples = 3
gain = 10 # 用于 vizDiff 的增益

if os.path.exists(datapath):
    # 遍历数据目录下的子文件夹 (每个文件夹通常是一个序列)
    subdirs = [d for d in os.listdir(datapath) if os.path.isdir(os.path.join(datapath, d))]
    
    for item in subdirs[:5]: # 示例仅处理第一个子目录
        datapath_item = os.path.join(datapath, item)
        print(f"[Processing] {datapath_item}")
        
        # 使用 TianmoucDataReader 读取实拍数据
        dataset = TianmoucDataReader(datapath_item, N=1, strict=True, _correct_sd_upsampler=True)
        print(f"Total samples: {len(dataset)}")
        
        for i in range(min(num_samples, len(dataset))):
            sample = dataset[i]
            # sample['rawDiff'] shape: (3, T, H, W) -> (TD, SDL, SDR, T, 160, 160)
            # 我们取 T=0 的这一帧
            raw_tsd = torch.Tensor(sample['rawDiff']).to(device) # (3, T, 160, 160)
            
            # 提取 SDL (index 1) 和 SDR (index 2)
            sd_single = raw_tsd[1:3, 0, :, :]/128.0 # (2, 160, 160)
            
            # 1. 预处理: 上采样至 320x640 (或模型训练时的分辨率)
            # 模仿训练时的输入构造
            sd_input = sd_single.unsqueeze(0).unsqueeze(0) # [1, 1, 2, 160, 160]
            # 假设目标分辨率是 320x640
            sd_up = upsample_sd(sd_input, sensor_height=320, sensor_width=640).squeeze(1) # [1, 2, 320, 640]
            
            # 2. 推理
            with torch.no_grad():
                pred_gray = model(sd_up) # [1, 1, 320, 640]
            
            # 3. 可视化
            # 输入 SD 可视化 (叠加 SDL 和 SDR)
            sd_viz = vizDiff(sd_single.mean(dim=0).cpu(), gain=gain, bg_color='black').numpy()/128.0
            
            # 预测结果归一化
            pred_viz = normalize_for_viz(pred_gray[0, 0].cpu().numpy())
            
            # 绘图对比
            plt.figure(figsize=(12, 6))
            plt.subplot(1, 2, 1)
            plt.imshow(sd_viz)
            plt.title(f"Input SD (Item: {item}, Sample: {i})")
            plt.axis('off')
            
            plt.subplot(1, 2, 2)
            plt.imshow(pred_viz, cmap='gray')
            plt.title("Reconstructed Gray")
            plt.axis('off')
            
            plt.tight_layout()
            plt.show()
else:
    print("Skip real data inference because path does not exist.")